# V20 pilot run 2 (validation only): pilot stage on the SAME teacher + K-prune masks from gouravcseai/v20-pilot (teacher sha256 verified). Tag v20-pilot-freeze-fix1

In [ ]:
%%bash
set -e
exec > >(tee -a /kaggle/working/run.log) 2>&1
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v20-pilot-freeze-fix1 && git log --oneline -1 | tee /kaggle/working/commit.txt
O=/kaggle/working/v20; mkdir -p $O
find /kaggle/input -maxdepth 6 2>/dev/null | head -80 > $O/input_listing.txt
META=$(find /kaggle/input -name ft_meta.json 2>/dev/null | head -1); echo "ft_meta: $META" | tee $O/input_root.txt
IN=$(dirname $(dirname $META)); echo "IN=$IN" | tee -a $O/input_root.txt
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | tee $O/gpu.txt
pip install -q -r requirements-ci.txt && pip install -q -e . --no-deps
pip uninstall -y -q torchvision torchaudio || true
python scripts/preflight.py check | tee $O/preflight.txt
python -c "import transformers,torch;assert transformers.__version__=='4.51.3';print(torch.__version__,transformers.__version__)" | tee $O/env.txt
# teacher from the first kernel's output: sha256 must equal the value recorded in PROJECT_STATE (amendment 2)
SHA=e19f3b747f4a4cadd96c4d4849b74e42222325eadc393b0149dd6d590350c901
echo "$SHA  $IN/ft/mnli/pytorch_model.bin" | sha256sum -c - || { echo TEACHER_SHA_MISMATCH; exit 1; }
cp -r $IN/ft /tmp/ft; cp $IN/ft/ft_meta.json $O/ft_meta.json
PYTHONPATH=scripts python scripts/v20_pilot.py --ckpt /tmp/ft/mnli --kprune-masks $IN/bl/kprune_masks.npz \
  --index-dir $IN/inputs/indices --output-dir $O/pilot > $O/pilot.log 2>&1 || { grep -v -i "warn\|xet" $O/pilot.log | tail -30; echo PILOT_FAILED; }
grep -v -i "warn\|xet" $O/pilot.log | cut -c1-230 | tail -50
ls $O/pilot | wc -l
cp /kaggle/working/run.log $O/run.log; cd /kaggle/working && tar czf v20_pilot2_results.tgz v20 commit.txt 2>/dev/null || true
ls -la /kaggle/working
